# Exploratory Data Analysis — Retail Sales Data
**Objective:** Uncover patterns, customer behaviour trends, and actionable business insights from a retail sales dataset.

**Dataset used:** *(fill in the exact Kaggle dataset name + link here once you've downloaded it)*

**Tech stack:** Python, pandas, matplotlib, seaborn


## Step 0 — Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Notebook display settings
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', None)


## Step 1 — Load Dataset & Initial Inspection
Replace `'retail_sales.csv'` with your actual filename. If your file is an Excel file
(common for the "Superstore" dataset), use `pd.read_excel()` instead.


In [ ]:
# --- Load the data ---
df = pd.read_csv('retail_sales.csv', encoding='latin1')   # try 'latin1' if utf-8 throws an error
# df = pd.read_excel('retail_sales.xlsx')  # use this line instead if the file is .xlsx

# --- Shape ---
print("Shape of dataset (rows, columns):", df.shape)

# --- Preview ---
df.head()


In [ ]:
# --- Column names and dtypes ---
df.info()


In [ ]:
# --- Null value check ---
null_summary = df.isnull().sum().sort_values(ascending=False)
null_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
null_report = pd.DataFrame({'missing_count': null_summary, 'missing_pct': null_pct.round(2)})
null_report[null_report['missing_count'] > 0]


**Observation (fill in):**
- Note which columns have missing values and roughly what % is missing.
- Note any dtype issues (e.g. a date column stored as text/object — you'll fix this below).


In [ ]:
# --- Fix common dtype issues ---
# Rename this to your actual date column name (e.g. 'Order Date')
date_col = 'Order Date'
if date_col in df.columns:
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    print(df[date_col].dtype)

# Drop exact duplicate rows if any
dupes = df.duplicated().sum()
print(f"Duplicate rows found: {dupes}")
df = df.drop_duplicates()


## Step 2 — Descriptive Statistics
Mean, median, mode, and standard deviation for every numerical column.


In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
print("Numerical columns:", num_cols)

desc = df[num_cols].describe().T
desc['median'] = df[num_cols].median()
desc['mode'] = df[num_cols].mode().iloc[0]
desc[['mean', 'median', 'mode', 'std', 'min', 'max']]


**Observation (fill in):** Which numeric columns are most spread out (high std)?
Do mean and median diverge a lot anywhere (sign of skew / outliers)?


## Step 3 — Time Series Analysis
Monthly and quarterly sales trends. Adjust `date_col` and `sales_col` to match your dataset.


In [ ]:
date_col = 'Order Date'   # <-- change if different
sales_col = 'Sales'       # <-- change if different

ts_df = df.dropna(subset=[date_col]).copy()
ts_df.set_index(date_col, inplace=True)

# --- Monthly sales trend ---
monthly_sales = ts_df[sales_col].resample('M').sum()

plt.figure(figsize=(12, 5))
monthly_sales.plot(marker='o')
plt.title('Monthly Sales Trend')
plt.xlabel('Month')
plt.ylabel('Total Sales')
plt.tight_layout()
plt.show()


In [ ]:
# --- Quarterly sales trend ---
quarterly_sales = ts_df[sales_col].resample('Q').sum()

plt.figure(figsize=(10, 5))
quarterly_sales.plot(kind='bar', color='steelblue')
plt.title('Quarterly Sales Trend')
plt.xlabel('Quarter')
plt.ylabel('Total Sales')
plt.tight_layout()
plt.show()


**Observation (fill in):** Is there seasonality (e.g. spikes around November/December —
holiday shopping)? Is the overall trend growing, flat, or declining?


## Step 4 — Customer Demographics
Age group distribution and gender breakdown. **Note:** the classic Superstore dataset does
*not* include age/gender — if yours doesn't either, either (a) pick a dataset that has
customer demographics (search "retail sales with customer demographics kaggle"), or
(b) substitute this section with **Segment** / **Region** breakdown, which the code below
also shows as a fallback.


In [ ]:
if 'Age' in df.columns:
    bins = [0, 18, 25, 35, 45, 55, 65, 100]
    labels = ['<18', '18-24', '25-34', '35-44', '45-54', '55-64', '65+']
    df['Age Group'] = pd.cut(df['Age'], bins=bins, labels=labels, right=False)

    plt.figure(figsize=(10, 5))
    sns.countplot(data=df, x='Age Group', order=labels, palette='viridis')
    plt.title('Customer Distribution by Age Group')
    plt.xlabel('Age Group')
    plt.ylabel('Number of Customers')
    plt.tight_layout()
    plt.show()
else:
    print("No 'Age' column found — see markdown note above for alternatives.")


In [ ]:
if 'Gender' in df.columns:
    plt.figure(figsize=(6, 6))
    df['Gender'].value_counts().plot(kind='pie', autopct='%1.1f%%', startangle=90)
    plt.title('Gender Breakdown')
    plt.ylabel('')
    plt.tight_layout()
    plt.show()
elif 'Segment' in df.columns:
    # Fallback for Superstore-style datasets
    plt.figure(figsize=(6, 6))
    df['Segment'].value_counts().plot(kind='pie', autopct='%1.1f%%', startangle=90)
    plt.title('Customer Segment Breakdown (fallback for Gender)')
    plt.ylabel('')
    plt.tight_layout()
    plt.show()


**Observation (fill in):** Which age group / segment brings in the most customers or revenue?
Any imbalance worth flagging?


## Step 5 — Product Analysis
Top 10 best-selling products, and revenue by product category.


In [ ]:
product_col = 'Product Name'   # <-- change if different
sales_col = 'Sales'

top_products = df.groupby(product_col)[sales_col].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 6))
top_products.sort_values().plot(kind='barh', color='coral')
plt.title('Top 10 Best-Selling Products (by Revenue)')
plt.xlabel('Total Sales')
plt.ylabel('Product')
plt.tight_layout()
plt.show()


In [ ]:
category_col = 'Category'   # <-- change if different

revenue_by_category = df.groupby(category_col)[sales_col].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
revenue_by_category.plot(kind='bar', color='teal')
plt.title('Revenue by Product Category')
plt.xlabel('Category')
plt.ylabel('Total Sales')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


**Observation (fill in):** Which single product dominates? Is revenue concentrated in one
category (80/20 pattern) or fairly spread out?


## Step 6 — Correlation Heatmap

In [ ]:
corr = df[num_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix — Numerical Variables')
plt.tight_layout()
plt.show()


**Observation (fill in):** Which variables move together strongly (e.g. Sales & Profit,
or Discount & Profit negatively)? Anything counter-intuitive?


## Step 7 — Additional Visualisation (Non-Obvious Insight)
Example below: **discount vs. profit** — a very common "hidden" business risk in retail
data (heavy discounting often erodes or reverses profit). Feel free to swap this for a
different angle relevant to your dataset — e.g. day-of-week sales pattern, shipping mode
vs. delivery delay, regional profitability, etc.


In [ ]:
discount_col = 'Discount'   # <-- change if different
profit_col = 'Profit'       # <-- change if different

if discount_col in df.columns and profit_col in df.columns:
    plt.figure(figsize=(9, 6))
    sns.scatterplot(data=df, x=discount_col, y=profit_col, alpha=0.4, hue=category_col if category_col in df.columns else None)
    plt.axhline(0, color='red', linestyle='--', linewidth=1)
    plt.title('Discount vs. Profit')
    plt.xlabel('Discount')
    plt.ylabel('Profit')
    plt.tight_layout()
    plt.show()
else:
    # Fallback: day-of-week sales pattern
    ts_df['DayOfWeek'] = ts_df.index.day_name()
    dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    dow_sales = ts_df.groupby('DayOfWeek')[sales_col].sum().reindex(dow_order)

    plt.figure(figsize=(9, 5))
    dow_sales.plot(kind='bar', color='purple')
    plt.title('Total Sales by Day of Week')
    plt.tight_layout()
    plt.show()


**Observation (fill in):** What's the non-obvious takeaway here? (e.g. "Discounts above
X% consistently push profit negative in Category Y" or "Weekend sales are Z% higher than
weekday sales.")


## Step 8 — Conclusion & Business Recommendations

**Summary of key findings:**
1. *(fill in)*
2. *(fill in)*
3. *(fill in)*

**Actionable recommendations:**
1. *(e.g. "Increase inventory for [top product] ahead of [peak month] to avoid stockouts.")*
2. *(e.g. "Cap discounts at X% in [category] where discounting is eroding profit margins.")*
3. *(e.g. "Target marketing spend at [age group / segment] which drives disproportionate revenue.")*
